**Run evidence** · task `00_preflight_checks` · run `466901783395814` · result **SUCCESS** · executed 2026-10-19 17:04 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `_helper/00_preflight_checks`.

# Preflight Checks — Validate environment before pipeline execution

Verifies **catalog permissions**, **SQL warehouse availability**, and **governed-tag
support** so problems surface immediately instead of halfway through the pipeline.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
from databricks.sdk import WorkspaceClient

print("=" * 70)
print("  PREFLIGHT CHECKS")
print(f"  Catalog:      {CFG.catalog}")
print(f"  Schema:       {CFG.schema}")
print(f"  Warehouse ID: {CFG.warehouse_id}")
print("=" * 70)

results = []  # list of (name, passed_bool)

# ─────────────────────────────────────────────────────────────────────
# CHECK 1 — Catalog & schema permissions
# ─────────────────────────────────────────────────────────────────────
print("\n── [1/4] Catalog & schema permissions ──")

catalog_ok = True

# 1a. USE CATALOG
try:
    spark.sql(f"USE CATALOG `{CFG.catalog}`")
    print(f"  ✓ USE CATALOG `{CFG.catalog}`")
except Exception as e:
    print(f"  ✗ USE CATALOG `{CFG.catalog}` — FAILED")
    print(f"    {str(e).splitlines()[0]}")
    catalog_ok = False

# 1b. CREATE SCHEMA (or verify it already exists)
schema_ok = False
if catalog_ok:
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG.catalog}`.`{CFG.schema}`")
        print(f"  ✓ CREATE SCHEMA IF NOT EXISTS `{CFG.catalog}`.`{CFG.schema}`")
        schema_ok = True
    except Exception:
        # No CREATE SCHEMA permission — check if schema already exists
        try:
            spark.sql(f"DESCRIBE SCHEMA `{CFG.catalog}`.`{CFG.schema}`")
            print(f"  ~ Schema `{CFG.schema}` already exists (no CREATE SCHEMA permission, but not needed)")
            schema_ok = True
        except Exception as e2:
            print(f"  ✗ Schema `{CFG.schema}` does not exist and cannot be created")
            print(f"    {str(e2).splitlines()[0]}")

# 1c. USE SCHEMA
if schema_ok:
    try:
        spark.sql(f"USE SCHEMA `{CFG.schema}`")
        print(f"  ✓ USE SCHEMA `{CFG.schema}` (catalog already set)")
    except Exception as e:
        print(f"  ✗ USE SCHEMA — {str(e).splitlines()[0]}")
        schema_ok = False

# 1d. Write test — create and immediately drop a tiny table
write_ok = False
if schema_ok:
    _test = f"`{CFG.catalog}`.`{CFG.schema}`.__preflight_write_test"
    try:
        spark.sql(f"CREATE TABLE {_test} (id INT) USING DELTA")
        spark.sql(f"DROP TABLE {_test}")
        print(f"  ✓ CREATE + DROP TABLE — write permissions OK")
        write_ok = True
    except Exception as e:
        print(f"  ✗ Write test — {str(e).splitlines()[0]}")

results.append(("Catalog & schema permissions", catalog_ok and schema_ok and write_ok))

# ─────────────────────────────────────────────────────────────────────
# CHECK 2 — SQL Warehouse usability
# ─────────────────────────────────────────────────────────────────────
print("\n── [2/4] SQL Warehouse ──")
wh_ok = False
try:
    w = WorkspaceClient()
    wh = w.warehouses.get(CFG.warehouse_id)
    state = str(wh.state).split(".")[-1]
    print(f"  ✓ Found warehouse \"{wh.name}\"")
    print(f"    State: {state}  |  Size: {wh.cluster_size}  |  Type: {wh.warehouse_type}")
    if state in ("RUNNING", "STOPPED"):
        print(f"  ✓ Warehouse is usable (state={state}, auto-starts on query)")
        wh_ok = True
    elif state == "DELETED":
        print(f"  ✗ Warehouse is DELETED — cannot be used")
    else:
        print(f"  ~ Warehouse state is {state} — may need attention")
        wh_ok = True  # transitional states may resolve
except Exception as e:
    print(f"  ✗ Warehouse check FAILED — {str(e).splitlines()[0]}")

results.append(("SQL Warehouse", wh_ok))

# ─────────────────────────────────────────────────────────────────────
# CHECK 3 — Governed tags (create + delete a test tag on the schema)
# ─────────────────────────────────────────────────────────────────────
print("\n── [3/4] Governed tags ──")
tag_ok = False
tag_key = "_preflight_test"
if schema_ok:
    try:
        spark.sql(
            f"ALTER SCHEMA `{CFG.catalog}`.`{CFG.schema}` "
            f"SET TAGS ('{tag_key}' = 'ok')"
        )
        print(f"  ✓ SET TAGS on schema — OK")
        spark.sql(
            f"ALTER SCHEMA `{CFG.catalog}`.`{CFG.schema}` "
            f"UNSET TAGS ('{tag_key}')"
        )
        print(f"  ✓ UNSET TAGS on schema — cleaned up")
        tag_ok = True
    except Exception as e:
        print(f"  ✗ Governed tags — {str(e).splitlines()[0]}")
else:
    print(f"  ⊘ Skipped (schema not available)")

results.append(("Governed tags", tag_ok))

# ─────────────────────────────────────────────────────────────────────
# CHECK 4 — Discover domains API + capacity (needed by 09a_create_domains)
# Read-only: lists domains (paginated) and checks there is room for the
# 3 domains/subdomains the demo creates. The workspace limit observed is
# 300 domains; above it, POST /api/2.0/domains returns ResourceExhausted.
# ─────────────────────────────────────────────────────────────────────
print("\n── [4/4] Discover domains capacity ──")
DOMAIN_LIMIT = 300
DEMO_DOMAIN_TAGS = [
    "SAP_Purchasing",
    "SAP_Purchasing/Sourcing_Supplier_Risk",
    "SAP_Purchasing/Purchase_Order_Analytics",
]
domains_ok = False
try:
    w = WorkspaceClient()
    _hdrs = {"Accept": "application/json"}
    _all, _tok = [], None
    while True:
        _q = {"page_size": 100}
        if _tok:
            _q["page_token"] = _tok
        _r = w.api_client.do("GET", "/api/2.0/domains", query=_q, headers=_hdrs)
        _all += _r.get("domains", [])
        _tok = _r.get("next_page_token")
        if not _tok:
            break
    existing_tags = {d.get("tag_key") for d in _all}
    to_create = [t for t in DEMO_DOMAIN_TAGS if t not in existing_tags]
    free = DOMAIN_LIMIT - len(_all)
    print(f"  ✓ Domains API reachable — {len(_all)} domains exist (limit {DOMAIN_LIMIT})")
    print(f"    Demo domains already present: {len(DEMO_DOMAIN_TAGS) - len(to_create)}/{len(DEMO_DOMAIN_TAGS)}")
    if len(to_create) <= free:
        print(f"  ✓ Capacity OK — need {len(to_create)}, {free} free")
        domains_ok = True
    else:
        print(f"  ✗ Not enough domain capacity — need {len(to_create)}, only {max(free, 0)} free")
        print(f"    09a_create_domains would fail with ResourceExhausted.")
        print(f"    A Discover curator must delete unused domains, then re-run.")
except Exception as e:
    print(f"  ✗ Domains API check FAILED — {str(e).splitlines()[0]}")
    print(f"    The 'Domains and Discover Page' preview may not be enabled.")

results.append(("Discover domains capacity", domains_ok))

# ─────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  SUMMARY")
print("=" * 70)
all_ok = True
for name, passed in results:
    icon = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {icon}  {name}")
    if not passed:
        all_ok = False

# Build a report string that dbutils.notebook.exit returns to the caller.
report_lines = []
for name, passed in results:
    icon = "PASS" if passed else "FAIL"
    report_lines.append(f"  {icon}: {name}")
report = "\n".join(report_lines)

if all_ok:
    print(f"\n✅ All preflight checks passed — safe to proceed.")
    dbutils.notebook.exit("SUCCESS\n" + report)
else:
    failed = [n for n, p in results if not p]
    summary = f"FAILED: {', '.join(failed)}"
    print(f"\n❌ {summary}")
    print("   Fix the issues above before running the pipeline.")
    dbutils.notebook.exit(summary + "\n" + report)

Notebook exited: SUCCESS
  PASS: Catalog & schema permissions
  PASS: SQL Warehouse
  PASS: Governed tags
  PASS: Discover domains capacity
